# Web Skin Disease — EfficientNet-B0 학습
## Original vs Augmented 비교

이 Notebook은 Google Drive에 있는 압축파일을 **먼저 자동으로 압축 해제**한 뒤,
`original`과 `augmented` 데이터를 각각 별도의 EfficientNet-B0 모델로 학습합니다.

### Google Drive 입력 위치

```text
MyDrive/
└── web_skin_dataset/
    └── web_skin_processed.zip
```

압축 해제 후 목표 구조:

```text
web_skin_processed/
├── original/
│   ├── train/
│   ├── val/
│   └── test/
│
└── augmented/
    ├── train/
    ├── val/
    └── test/
```

### 학습 비교

**Original 모델**
- `original/train`
- `original/val`
- `original/test`

**Augmented 모델**
- `augmented/train`
- `augmented/val`
- `augmented/test`

`augmented/train`은 **원본 + 증강** 데이터이고,
`augmented/val`, `augmented/test`는 **원본을 그대로 복사한 데이터**입니다.

두 모델 모두 동일한 클래스와 동일한 평가 기준으로 비교합니다.


In [ ]:
# ============================================================
# 1. Google Drive 연결
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive 연결 완료")


In [ ]:
# ============================================================
# 2. Drive의 ZIP 위치 설정
# ============================================================

from pathlib import Path
import zipfile
import shutil
import os
import random
import json
import time

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/web_skin_dataset"
)

ZIP_PATH = (
    DRIVE_DATA_DIR
    / "web_skin_processed.zip"
)

EXTRACT_ROOT = Path(
    "/content/web_skin_dataset"
)

print("ZIP 경로:")
print(ZIP_PATH)

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP 파일을 찾지 못했습니다.\n\n{ZIP_PATH}\n\n"
        "Google Drive에서 파일명이 web_skin_processed.zip인지 확인하세요."
    )

print("ZIP 파일 확인 완료")


## 3. ZIP 압축 해제

Colab의 `/content`에 압축을 풀어 학습합니다.

기존 압축 해제 결과가 남아 있어도 데이터가 섞이지 않도록
이번 실행에서는 `/content/web_skin_dataset`을 먼저 삭제하고 다시 압축 해제합니다.

또한 ZIP 내부가

```text
web_skin_processed/original/...
web_skin_processed/augmented/...
```

형태인지,

또는

```text
original/...
augmented/...
```

형태인지 자동으로 찾아서 처리합니다.


In [ ]:
# ============================================================
# 3. ZIP 압축 해제
# ============================================================

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("ZIP 압축 해제 시작")
print("=" * 70)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)

print("압축 해제 완료")

# ------------------------------------------------------------
# original / augmented 폴더 자동 탐색
# ------------------------------------------------------------

def find_dataset_root(base):
    base = Path(base)

    candidates = []

    # 직접 아래에 있는 경우
    if (
        (base / "original").is_dir()
        and
        (base / "augmented").is_dir()
    ):
        candidates.append(base)

    # 하위 폴더에 web_skin_processed가 있는 경우
    for p in base.rglob("*"):
        if p.is_dir():
            if (
                (p / "original").is_dir()
                and
                (p / "augmented").is_dir()
            ):
                candidates.append(p)

    # 중복 제거
    unique = []
    seen = set()

    for p in candidates:
        p = p.resolve()

        if str(p) not in seen:
            seen.add(str(p))
            unique.append(p)

    if not unique:
        raise FileNotFoundError(
            "압축 해제 후 original/augmented 폴더를 찾지 못했습니다."
        )

    return unique[0]


DATASET_ROOT = find_dataset_root(
    EXTRACT_ROOT
)

ORIGINAL_ROOT = (
    DATASET_ROOT / "original"
)

AUGMENTED_ROOT = (
    DATASET_ROOT / "augmented"
)

print()
print("실제 데이터셋 위치:")
print(DATASET_ROOT)

print()
print("Original:")
print(ORIGINAL_ROOT)

print()
print("Augmented:")
print(AUGMENTED_ROOT)


In [ ]:
# ============================================================
# 4. 폴더 구조 확인
# ============================================================

print()
print("=" * 70)
print("데이터셋 구조 확인")
print("=" * 70)

for root_name, root in [
    ("original", ORIGINAL_ROOT),
    ("augmented", AUGMENTED_ROOT),
]:
    print()
    print(f"[{root_name}]")

    for split in ["train", "val", "test"]:
        split_dir = root / split

        if not split_dir.exists():
            print(f"  [오류] {split_dir} 없음")
            continue

        classes = [
            p.name
            for p in split_dir.iterdir()
            if p.is_dir()
        ]

        print(
            f"  {split}: {classes}"
        )


In [ ]:
# ============================================================
# 5. 라이브러리 설치 / GPU 확인
# ============================================================

!pip install -q scikit-learn seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if tf.config.list_physical_devices("GPU"):
    print("GPU 사용 가능")
else:
    print("주의: GPU가 없습니다. Colab에서 GPU 런타임을 선택하세요.")


In [ ]:
# ============================================================
# 6. 학습 설정
# ============================================================

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
SEED = 42

AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = [
    "건선",
    "아토피",
    "여드름",
    "정상",
    "주사"
]

NUM_CLASSES = len(CLASS_NAMES)

print("클래스:", CLASS_NAMES)
print("클래스 수:", NUM_CLASSES)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)


## 7. 데이터 개수 검증

예상 구조는 다음과 같습니다.

### Original

```text
train = 클래스당 720
val   = 클래스당 100
test  = 클래스당 80
```

### Augmented

```text
train = 클래스당 1,440
val   = 클래스당 100
test  = 클래스당 80
```

특히 `augmented/val`과 `augmented/test`는 **증강하지 않은 원본 복사본**입니다.


In [ ]:
# ============================================================
# 7. 데이터 개수 검증
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

def count_images(folder):
    folder = Path(folder)

    if not folder.exists():
        return 0

    return sum(
        1
        for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def show_counts(root, title):
    print()
    print("=" * 70)
    print(title)
    print("=" * 70)

    total = 0

    for split in ["train", "val", "test"]:
        split_total = 0

        print(f"\n[{split}]")

        for class_name in CLASS_NAMES:
            folder = (
                root
                / split
                / class_name
            )

            count = count_images(folder)

            split_total += count
            total += count

            print(
                f"  {class_name:<8} : {count:4d}"
            )

        print(
            f"  합계 : {split_total:,}"
        )

    print(
        f"\n전체 : {total:,}"
    )


show_counts(
    ORIGINAL_ROOT,
    "ORIGINAL DATASET"
)

show_counts(
    AUGMENTED_ROOT,
    "AUGMENTED DATASET"
)


In [ ]:
# ============================================================
# 8. TensorFlow Dataset 생성
# ============================================================

def create_datasets(root):
    root = Path(root)

    train_ds = tf.keras.utils.image_dataset_from_directory(
        root / "train",
        labels="inferred",
        label_mode="categorical",
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        root / "val",
        labels="inferred",
        label_mode="categorical",
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        root / "test",
        labels="inferred",
        label_mode="categorical",
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    return (
        train_ds.prefetch(AUTOTUNE),
        val_ds.prefetch(AUTOTUNE),
        test_ds.prefetch(AUTOTUNE),
    )


original_train_ds, original_val_ds, original_test_ds = (
    create_datasets(ORIGINAL_ROOT)
)

augmented_train_ds, augmented_val_ds, augmented_test_ds = (
    create_datasets(AUGMENTED_ROOT)
)

print("Original Dataset 준비 완료")
print("Augmented Dataset 준비 완료")


## 9. EfficientNet-B0 모델

ImageNet pretrained EfficientNet-B0를 사용합니다.

초기 실험에서는 backbone을 동결하고 classification head를 학습합니다.

```text
224 × 224 × 3
      ↓
EfficientNet-B0
      ↓
Global Average Pooling
      ↓
Dropout
      ↓
Dense(5)
      ↓
Softmax
```


In [ ]:
# ============================================================
# 9. EfficientNet-B0 모델 생성
# ============================================================

def build_model():

    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(
            224,
            224,
            3
        )
    )

    base_model.trainable = False

    inputs = keras.Input(
        shape=(
            224,
            224,
            3
        )
    )

    x = base_model(
        inputs,
        training=False
    )

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(
        0.3
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-4
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


model = build_model()

model.summary()


In [ ]:
# ============================================================
# 10. 결과 저장 경로
# ============================================================

RESULT_ROOT = Path(
    "/content/web_skin_training_results"
)

if RESULT_ROOT.exists():
    shutil.rmtree(RESULT_ROOT)

ORIGINAL_RESULT_DIR = (
    RESULT_ROOT / "original"
)

AUGMENTED_RESULT_DIR = (
    RESULT_ROOT / "augmented"
)

ORIGINAL_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUGMENTED_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("결과 저장 위치:")
print(RESULT_ROOT)


# 11. 학습 + 평가 함수

Original과 Augmented 모델에 동일한 학습 조건을 적용합니다.

- EfficientNet-B0
- ImageNet pretrained
- 224×224
- Batch 32
- 15 Epoch
- Adam
- Learning rate 1e-4

각 모델에 대해 다음을 저장합니다.

- best model
- Accuracy 그래프
- Loss 그래프
- Confusion Matrix
- Classification Report
- 학습 History
- 결과 JSON


In [ ]:
# ============================================================
# 11. 학습 및 평가
# ============================================================

def train_and_evaluate(
    model_name,
    train_ds,
    val_ds,
    test_ds,
    result_dir
):

    print()
    print("=" * 75)
    print(f"{model_name} 모델 학습 시작")
    print("=" * 75)

    model = build_model()

    best_model_path = (
        result_dir / "best_model.keras"
    )

    checkpoint = keras.callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    )

    start_time = time.time()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[checkpoint],
        verbose=1
    )

    training_time = (
        time.time() - start_time
    )

    print()
    print(
        f"{model_name} 학습 완료"
    )

    print(
        f"학습 시간: "
        f"{training_time / 60:.2f}분"
    )

    # --------------------------------------------------------
    # Best model 로드
    # --------------------------------------------------------

    best_model = keras.models.load_model(
        best_model_path
    )

    # --------------------------------------------------------
    # Test 평가
    # --------------------------------------------------------

    test_loss, test_accuracy = (
        best_model.evaluate(
            test_ds,
            verbose=1
        )
    )

    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    plt.figure(
        figsize=(8, 6)
    )

    plt.plot(
        history.history["accuracy"],
        label="Train Accuracy"
    )

    plt.plot(
        history.history["val_accuracy"],
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(
        f"{model_name} Accuracy"
    )

    plt.legend()
    plt.grid()

    plt.tight_layout()

    plt.savefig(
        result_dir / "accuracy.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    plt.figure(
        figsize=(8, 6)
    )

    plt.plot(
        history.history["loss"],
        label="Train Loss"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(
        f"{model_name} Loss"
    )

    plt.legend()
    plt.grid()

    plt.tight_layout()

    plt.savefig(
        result_dir / "loss.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    y_true = []
    y_pred = []

    for images, labels in test_ds:

        predictions = best_model.predict(
            images,
            verbose=0
        )

        predicted = np.argmax(
            predictions,
            axis=1
        )

        actual = np.argmax(
            labels.numpy(),
            axis=1
        )

        y_pred.extend(
            predicted.tolist()
        )

        y_true.extend(
            actual.tolist()
        )

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # --------------------------------------------------------
    # Confusion Matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    plt.figure(
        figsize=(9, 7)
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(
        f"{model_name} Confusion Matrix"
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.yticks(
        rotation=0
    )

    plt.tight_layout()

    plt.savefig(
        result_dir / "confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # --------------------------------------------------------
    # Classification metrics
    # --------------------------------------------------------

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES)),
            zero_division=0
        )
    )

    report_df = pd.DataFrame({
        "class": CLASS_NAMES,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support
    })

    macro_precision = float(
        precision.mean()
    )

    macro_recall = float(
        recall.mean()
    )

    macro_f1 = float(
        f1.mean()
    )

    print()
    print(
        report_df.to_string(
            index=False
        )
    )

    report_df.to_csv(
        result_dir / "classification_report.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # History 저장
    # --------------------------------------------------------

    history_dict = {
        key: [
            float(v)
            for v in values
        ]
        for key, values
        in history.history.items()
    }

    with open(
        result_dir / "training_history.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            history_dict,
            f,
            ensure_ascii=False,
            indent=4
        )

    # --------------------------------------------------------
    # 결과 저장
    # --------------------------------------------------------

    best_val_accuracy = max(
        history.history["val_accuracy"]
    )

    best_epoch = int(
        np.argmax(
            history.history["val_accuracy"]
        ) + 1
    )

    result = {
        "model": model_name,
        "architecture": "EfficientNet-B0",
        "pretrained": "ImageNet",
        "image_size": [224, 224],
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": 1e-4,
        "best_epoch": best_epoch,
        "best_val_accuracy": float(
            best_val_accuracy
        ),
        "test_accuracy": float(
            test_accuracy
        ),
        "test_loss": float(
            test_loss
        ),
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "training_time_minutes": (
            float(training_time / 60)
        ),
        "classes": CLASS_NAMES
    }

    with open(
        result_dir / "results.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=4
        )

    print()
    print("=" * 75)
    print(f"{model_name} 결과")
    print("=" * 75)

    print(
        f"Best Val Accuracy : "
        f"{best_val_accuracy * 100:.2f}%"
    )

    print(
        f"Test Accuracy     : "
        f"{test_accuracy * 100:.2f}%"
    )

    print(
        f"Macro F1          : "
        f"{macro_f1:.4f}"
    )

    return result


# 12. Original 모델 학습

`original/train`을 사용합니다.

클래스당 720장 × 5개 클래스 = **3,600장**입니다.

Validation은 클래스당 100장, Test는 클래스당 80장입니다.


In [ ]:
# ============================================================
# 12. ORIGINAL 모델 학습
# ============================================================

original_result = train_and_evaluate(
    model_name="Original",
    train_ds=original_train_ds,
    val_ds=original_val_ds,
    test_ds=original_test_ds,
    result_dir=ORIGINAL_RESULT_DIR
)


# 13. Augmented 모델 학습

`augmented/train`을 사용합니다.

클래스당:

```text
원본 720
+
증강 720
=
1,440장
```

즉 전체 Train은 **7,200장**입니다.

`augmented/val`과 `augmented/test`는 증강하지 않은 원본 복사본입니다.


In [ ]:
# ============================================================
# 13. AUGMENTED 모델 학습
# ============================================================

augmented_result = train_and_evaluate(
    model_name="Augmented",
    train_ds=augmented_train_ds,
    val_ds=augmented_val_ds,
    test_ds=augmented_test_ds,
    result_dir=AUGMENTED_RESULT_DIR
)


# 14. Original vs Augmented 비교

두 모델의 성능을 표로 비교합니다.

특히 확인할 지표:

- Test Accuracy
- Macro Precision
- Macro Recall
- Macro F1
- Best Validation Accuracy

그리고 두 모델의 Confusion Matrix를 함께 확인합니다.


In [ ]:
# ============================================================
# 14. Original vs Augmented 비교
# ============================================================

comparison_df = pd.DataFrame([
    {
        "Model": "Original",
        "Best Val Accuracy":
            original_result["best_val_accuracy"],
        "Test Accuracy":
            original_result["test_accuracy"],
        "Macro Precision":
            original_result["macro_precision"],
        "Macro Recall":
            original_result["macro_recall"],
        "Macro F1":
            original_result["macro_f1"],
        "Best Epoch":
            original_result["best_epoch"]
    },
    {
        "Model": "Augmented",
        "Best Val Accuracy":
            augmented_result["best_val_accuracy"],
        "Test Accuracy":
            augmented_result["test_accuracy"],
        "Macro Precision":
            augmented_result["macro_precision"],
        "Macro Recall":
            augmented_result["macro_recall"],
        "Macro F1":
            augmented_result["macro_f1"],
        "Best Epoch":
            augmented_result["best_epoch"]
    }
])

display(
    comparison_df.style.format({
        "Best Val Accuracy": "{:.4f}",
        "Test Accuracy": "{:.4f}",
        "Macro Precision": "{:.4f}",
        "Macro Recall": "{:.4f}",
        "Macro F1": "{:.4f}"
    })
)

comparison_df.to_csv(
    RESULT_ROOT / "original_vs_augmented.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
# ============================================================
# 15. 성능 차이
# ============================================================

acc_diff = (
    augmented_result["test_accuracy"]
    -
    original_result["test_accuracy"]
)

f1_diff = (
    augmented_result["macro_f1"]
    -
    original_result["macro_f1"]
)

print()
print("=" * 70)
print("성능 차이")
print("=" * 70)

print(
    f"Test Accuracy 차이 : "
    f"{acc_diff * 100:+.2f}%p"
)

print(
    f"Macro F1 차이       : "
    f"{f1_diff:+.4f}"
)

if (
    augmented_result["test_accuracy"]
    >
    original_result["test_accuracy"]
):

    print()
    print("→ Test Accuracy 기준: Augmented 모델 우세")

elif (
    augmented_result["test_accuracy"]
    <
    original_result["test_accuracy"]
):

    print()
    print("→ Test Accuracy 기준: Original 모델 우세")

else:

    print()
    print("→ Test Accuracy 동일")


# 16. 결과 파일 확인

학습 결과는 Colab의 `/content/web_skin_training_results`에 저장됩니다.

```text
web_skin_training_results/
├── original/
│   ├── best_model.keras
│   ├── accuracy.png
│   ├── loss.png
│   ├── confusion_matrix.png
│   ├── classification_report.csv
│   ├── training_history.json
│   └── results.json
│
├── augmented/
│   └── 동일한 구조
│
└── original_vs_augmented.csv
```


In [ ]:
# ============================================================
# 16. 결과 파일 확인
# ============================================================

print()
print("=" * 70)
print("결과 파일")
print("=" * 70)

for path in sorted(
    RESULT_ROOT.rglob("*")
):
    if path.is_file():
        print(
            "OK:",
            path.relative_to(RESULT_ROOT)
        )


# 17. Google Drive에 결과 저장

Colab 세션이 종료되어도 학습 결과를 보존할 수 있도록
결과 폴더를 Google Drive의 `web_skin_dataset` 안에 복사하고 ZIP으로 압축합니다.


In [ ]:
# ============================================================
# 17. 결과를 Google Drive에 저장
# ============================================================

DRIVE_RESULT_DIR = (
    DRIVE_DATA_DIR
    / "web_skin_training_results"
)

if DRIVE_RESULT_DIR.exists():
    shutil.rmtree(
        DRIVE_RESULT_DIR
    )

shutil.copytree(
    RESULT_ROOT,
    DRIVE_RESULT_DIR
)

zip_base = (
    DRIVE_DATA_DIR
    / "web_skin_training_results"
)

drive_zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=DRIVE_RESULT_DIR.parent,
    base_dir=DRIVE_RESULT_DIR.name
)

print("결과 폴더:")
print(DRIVE_RESULT_DIR)

print()
print("결과 ZIP:")
print(drive_zip_path)


# 18. 완료

이제 이 Notebook을 실행하면:

1. Google Drive에서 `web_skin_processed.zip` 탐색
2. `/content`에 압축 해제
3. `original/augmented` 구조 자동 확인
4. 데이터 개수 확인
5. Original EfficientNet-B0 15 Epoch 학습
6. Augmented EfficientNet-B0 15 Epoch 학습
7. 두 모델 평가
8. Confusion Matrix / Classification Report 생성
9. Original vs Augmented 비교
10. 결과를 Google Drive에 저장

까지 한 번에 진행됩니다.
